In [1]:
from google.colab import userdata
import os

# Retrieve the Groq API key from your Colab secrets
try:
    groq_api_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = groq_api_key
    print("Groq API key successfully loaded into environment variables.")
except userdata.SecretNotFoundError:
    print("Error: The secret 'GROQ_API_KEY' was not found in your Colab secrets. Please add it in the Secrets tab (🔑 icon on the left).")

Groq API key successfully loaded into environment variables.


In [2]:
!pip install langmem langgraph langchain langchain-groq langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [langmem]


In [10]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.store.memory import InMemoryStore

MODEL = "openai/gpt-oss-120b"

def get_llm():
    return ChatGroq(model=MODEL, temperature=0)

def get_store():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    # The index tells the store how to embed items, so search is semantic
    return InMemoryStore(index={"dims": 384, "embed": embeddings})

def memory_text(item) -> str:
    """Memory tools store {"content": "..."}; the background manager stores
    {"kind": "Memory", "content": {"content": "..."}}. Handle both."""
    content = item.value.get("content")
    return content["content"] if isinstance(content, dict) else str(content)

In [5]:
from langchain.agents import create_agent

agent = create_agent(get_llm(), tools=[], system_prompt="You are a helpful personal assistant.")

agent.invoke({"messages": [("user", "Hi! I'm Priya. I'm vegetarian and I live in Chennai.")]})
result = agent.invoke({"messages": [("user", "Suggest a dinner place for me tonight.")]})
print(result["messages"][-1].content)

Sure! I can help you find a great spot, but I’ll need a little more information to make the recommendation truly fit your night. Could you let me know:

| Preference | Details to consider |
|------------|----------------------|
| **Location** | City or neighborhood you’ll be in (or a zip code) |
| **Cuisine** | Any type you’re craving (e.g., Italian, sushi, Mexican, vegan, etc.) |
| **Budget** | Rough price range per person (e.g., $‑$ (budget), $$ (mid‑range), $$$ (upscale)) |
| **Atmosphere** | Casual, romantic, family‑friendly, lively bar‑scene, quiet lounge, etc. |
| **Dietary needs** | Vegetarian, vegan, gluten‑free, allergies, etc. |
| **Group size** | Solo, couple, small group (2‑4), larger party (5+) |
| **Special occasion?** | Birthday, anniversary, business meeting, etc. |
| **Time** | Approximate dinner time (e.g., 6 pm, 8 pm) and how long you’d like to stay |

Just fill in whatever applies, and I’ll give you a tailored list of restaurants (including a short description, pric

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

llm = get_llm()
agent = create_agent(
    llm,
    tools=[],
    system_prompt="You are a helpful personal assistant.",
    checkpointer=InMemorySaver(),
    middleware=[
        # Once the thread passes 10 messages, replace older ones with a summary
        SummarizationMiddleware(model=llm, trigger=("messages", 10), keep=("messages", 4)),
    ],
)

thread_a = {"configurable": {"thread_id": "thread-a"}}
thread_b = {"configurable": {"thread_id": "thread-b"}}

agent.invoke({"messages": [("user", "Hi! I'm Priya. I'm vegetarian.")]}, thread_a)

r = agent.invoke({"messages": [("user", "What's my name and diet?")]}, thread_a)
print("same thread:", r["messages"][-1].content)   # remembers

r = agent.invoke({"messages": [("user", "What's my name and diet?")]}, thread_b)
print("new thread :", r["messages"][-1].content)   # forgets

same thread: Your name is **Priya**, and you follow a **vegetarian** diet.
new thread : I don’t have any information about your name or dietary preferences. If you’d like to share those details, I’ll be happy to keep them in mind for our conversation and help you with any related questions or suggestions!


In [8]:
from langmem import create_manage_memory_tool, create_search_memory_tool

store = get_store()
namespace = ("memories", "{user_id}")   # filled in at runtime, one space per user

agent = create_agent(
    get_llm(),
    tools=[
        create_manage_memory_tool(namespace=namespace),  # create / update / delete
        create_search_memory_tool(namespace=namespace),  # semantic search
    ],
    system_prompt=(
        "You are a helpful personal assistant with long-term memory.\n"
        "- When the user shares a lasting fact or preference, save it with manage_memory.\n"
        "- The `content` argument MUST be one plain-English sentence (a string). "
        "Never pass JSON, an object or a list.\n"
        "- Save one fact per call; make several calls if there are several facts.\n"
        "- Before answering anything personal, check search_memory first."
    ),
    store=store,
    checkpointer=InMemorySaver(),
)

def chat(text, thread_id):
    config = {"configurable": {"user_id": "priya", "thread_id": thread_id}}
    return agent.invoke({"messages": [("user", text)]}, config)["messages"][-1].content

# Session 1
chat("Hi! I'm Priya. I'm vegetarian, live in Chennai and hate spicy food.", "session-1")

# Session 2: a brand-new thread
print(chat("Suggest a dinner place for me tonight.", "session-2"))

for item in store.search(("memories", "priya"), query="food preferences"):
    print("MEMORY:", item.value["content"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Here are a few dinner spots in Chennai that cater to vegetarian diners and keep the heat mild, so you can enjoy a tasty, non‑spicy meal tonight:

| Restaurant | Neighborhood | Cuisine | Why it fits you |
|------------|--------------|---------|-----------------|
| **Mamagoto** | Nungambakkam / Alwarpet | Pan‑Asian (vegetarian‑friendly) | Large vegetarian menu, many dishes are lightly seasoned rather than fiery. |
| **The Brew Room** | Thiruvanmiyur | Continental / Café | Offers plenty of veggie pastas, salads, and grilled dishes with subtle flavors; no mandatory spice. |
| **Sowbagh – The Vegetarian Café** | Besant Nagar | Indian (South & North) | 100 % vegetarian, known for its gentle, aromatic preparations (no extra chilies). |
| **Annalakshmi** | Kotturpuram | South Indian (pure veg) | Traditional South Indian fare prepared without heavy heat; great for a relaxed dinner. |
| **The Green Table** | Adyar | Healthy/International | Focuses on wholesome vegetarian plates, soups, and bowls

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage
from langmem import create_memory_store_manager

llm = get_llm()
store = get_store()

manager = create_memory_store_manager(
    llm,
    namespace=("memories", "{user_id}"),
    store=store,
)

def chat(text, user_id="priya"):
    config = {"configurable": {"user_id": user_id}}

    # 1) Recall: put relevant memories into the prompt
    hits = store.search(("memories", user_id), query=text, limit=5)
    known = "\n".join(f"- {memory_text(h)}" for h in hits) or "- (nothing yet)"
    system = SystemMessage(f"You are a helpful assistant.\nWhat you know about the user:\n{known}")

    reply = llm.invoke([system, HumanMessage(text)]).content

    # 2) Learn: extract memories from this exchange
    manager.invoke(
        {"messages": [{"role": "user", "content": text},
                      {"role": "assistant", "content": reply}]},
        config=config,
    )
    return reply

chat("I'm Priya, I'm vegetarian and I live in Chennai.")
chat("Actually I've gone vegan this month.")   # should UPDATE, not duplicate
print(chat("What should I cook this weekend?"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hey Priya! 🌱 Since you’re looking for vegan ideas for the weekend and you’re in Chennai, I’ll suggest a mix of quick‑and‑easy dishes and a couple of slightly more involved recipes that make the most of the fresh, tropical ingredients you can find locally. Feel free to mix‑and‑match or adjust the spice levels to suit your taste.

---

## 🎉 Weekend Vegan Menu Ideas

| Meal | Dish | Why it fits Chennai | Key Ingredients | Approx. Time |
|------|------|---------------------|----------------|--------------|
| **Saturday Breakfast** | **Coconut‑Lemon Rice (Thengai Sadam)** | Coconut is abundant on the coast; the lemon adds a refreshing zing for the warm weather. | Cooked rice, fresh grated coconut, mustard seeds, curry leaves, green chilies, lemon juice, peanuts (optional) | 20 min |
| **Saturday Lunch** | **South Indian Tomato & Lentil Rasam with Millets** | Light, tangy, and perfect with a side of **ragi roti** (finger millet flatbread). | Toor dal, tomatoes, tamarind, rasam powder, mustar